# 🤝 GitHub 협업 워크플로우

**주제**: 여러 명이서 한 코드베이스를 같이 개발할 때, 어떻게 충돌 없이 작업하는가

| 섹션 | 내용 |
|------|------|
| 1. 왜 협업에 git이 필요한가 | "카톡으로 코드 주고받기" 의 한계 |
| 2. git vs GitHub | 자주 헷갈리는 둘의 차이 |
| 3. 처음 세팅 | clone, 인증(PAT/SSH), `git config` |
| 4. 혼자 vs 팀 작업 흐름 | 일상 cycle 비교 |
| 5. 브랜치 기초 | 왜 만드는지, 어떻게 만드는지 |
| 6. Pull Request 흐름 (GitHub Flow) | 팀 작업의 표준 사이클 |
| 7. 충돌(merge conflict) 해결 | 자주 만나지만 무섭지 않음 |
| 8. 일상 시나리오 8개 | 자주 마주치는 상황별 명령어 |
| 9. 좋은 커밋 습관 | 메시지 · `.gitignore` |
| 10. 자주 하는 실수 & 복구 | `reflog`, `reset`, `revert` |
| 11. 협업 에티켓 | 작은 PR · 친절한 리뷰 |


---
## 1. 왜 협업에 git이 필요한가

### 1.1 git 없이 협업한다면?

- 카톡으로 `main.py` 주고받음
- A가 보낸 버전에 B가 수정 → A도 자기 컴퓨터에서 수정 → 어느 게 최신?
- "이거 어제 거랑 뭐가 다르지?" 비교 안 됨
- 같은 줄을 동시에 고치면 한쪽 작업 통째로 손실

### 1.2 git이 해결하는 것

| 문제 | git의 해법 |
|------|----------|
| 누가/언제/뭘 바꿨는지 모름 | `commit` 단위로 작가 · 시간 · 변경 내역 기록 |
| 옛날 버전으로 못 돌아감 | `log`, `checkout`, `revert` |
| 동시 작업이 무서움 | `branch` 로 각자의 작업 공간 분리 |
| 합치는 게 무서움 | 자동 merge + 충돌만 사람이 해결 |
| 백업이 한 컴퓨터에만 | `push` 로 GitHub에 원격 백업 |


---
## 2. git vs GitHub — 자주 헷갈리는 둘

| 항목 | **git** | **GitHub** |
|------|---------|-----------|
| 무엇 | 버전 관리 **도구** (프로그램) | git 저장소 **호스팅 서비스** (웹사이트) |
| 어디서 동작 | 내 컴퓨터 (오프라인 OK) | 인터넷 (github.com) |
| 만든 곳 | Linus Torvalds (2005) | Microsoft 소유 (2018~) |
| 비유 | 워드의 변경 내역 추적 | 구글 드라이브 (공유 · 협업) |

```
┌─────────────────┐                 ┌──────────────────┐
│  내 컴퓨터       │  ←── push ───   │   GitHub         │
│  (로컬 repo)    │  ─── pull ──→   │   (원격 repo)    │
│  git 명령으로    │                 │   웹 UI · 협업 │
│  작업           │                 │   기능           │
└─────────────────┘                 └──────────────────┘
```

> 💡 GitHub 외에도 GitLab, Bitbucket 등 비슷한 호스팅 서비스가 있다. 모두 안에서 도는 건 똑같은 git.


---
## 3. 처음 세팅

### 3.1 내 정보 등록 (한 번만)

커밋 작성자 이름·이메일. GitHub 가입 이메일로 맞추면 GitHub에서 본인 프로필이랑 연결돼 보임.

```bash
git config --global user.name  "Hong Gildong"
git config --global user.email "gildong@example.com"

# 확인
git config --list | grep user
```

### 3.2 인증 — HTTPS + Personal Access Token (PAT)

GitHub은 2021년부터 비밀번호 인증을 막아둠. 두 가지 방법 중 하나:

| 방법 | 추천 상황 |
|------|---------|
| **HTTPS + PAT** | 처음 배울 때 가장 간단 |
| **SSH key** | 자주 push/pull 하는 메인 PC |

**PAT 발급**
1. https://github.com/settings/tokens → **Generate new token (classic)**
2. Scope에서 `repo` 체크 (private 까지 쓸거면)
3. 만료기간 설정 (90일 추천), 토큰 문자열 **복사 (한 번만 표시됨)**
4. push 시 비밀번호 자리에 이 토큰 입력

macOS는 keychain이 자동으로 저장해주므로 한 번만 입력하면 됨.

### 3.3 저장소 가져오기 — `clone`

```bash
# GitHub 페이지의 Code 버튼 → URL 복사
git clone https://github.com/user/repo.git
cd repo
```

`clone` 은 다음을 한 번에 한다:
- 빈 폴더 생성
- 원격 저장소 데이터 전부 다운로드
- `origin` 이라는 이름으로 원격 등록
- `main` 브랜치 체크아웃


---
## 4. 혼자 작업 vs 팀 작업 흐름

### 4.1 혼자 작업할 때 (이전 챕터 복습)

```bash
# 코드 수정
git add 수정한_파일들
git commit -m "Add login form"
git push
```

### 4.2 팀 작업할 때 — 두 가지 추가 규칙

1. **`main` 에 직접 커밋 금지.** 항상 **브랜치** 에서 작업
2. **작업 시작 전·후에 `git pull`** — 동료들의 최신 작업을 항상 가져온다

```bash
# (1) 시작 전: main 최신화
git checkout main
git pull

# (2) 새 브랜치에서 작업
git checkout -b feature/login

# 코드 수정 ... 커밋 여러 번
git add .
git commit -m "Add login form"

# (3) 푸시 — 첫 푸시는 -u 옵션
git push -u origin feature/login

# (4) GitHub 웹에서 Pull Request 열기

# (5) 머지된 뒤 정리
git checkout main
git pull
git branch -d feature/login          # 로컬 브랜치 정리
```

```
혼자 작업                     팀 작업
────────                     ─────────
edit                         git pull        ← 최신 받기
add                          git checkout -b ← 브랜치
commit                       edit / add / commit
push                         push -u
                             PR 열기
                             리뷰 · 머지
                             pull / branch -d
```

> ⚠️ 혼자 작업하던 습관으로 main에서 바로 커밋하면, 동료와 충돌이 거의 보장된다.


---
## 5. 브랜치 — 평행 우주

### 5.1 브랜치란?

브랜치는 **언제든 갈라낼 수 있는 평행 우주**다. `main` 의 어느 시점에서 갈라져 나와 자유롭게 커밋한 뒤, 다시 `main` 으로 합칠 수 있다.

```
main:     A───B───C───────────────G───
                   \             /
feature:            D───E───F───┘
```

- A, B, C, G : `main` 브랜치의 커밋
- D, E, F    : 작업용 브랜치의 커밋
- G          : 작업이 main에 합쳐진 머지 커밋

### 5.2 왜 브랜치를 쓰나?

- 미완성 코드가 `main` 에 들어가지 않음 → main은 항상 작동
- 여러 사람이 동시에 다른 일 진행 가능
- 안 좋으면 그냥 브랜치 통째로 버리면 됨

### 5.3 브랜치 명령어

```bash
# 현재 어디 있는지
git branch                 # 로컬 브랜치 목록 (* 가 현재)
git status                 # On branch main

# 새 브랜치 만들고 그쪽으로 이동
git checkout -b feature/login        # 옛 명령
git switch -c feature/login          # 권장 (git 2.23+)

# 기존 브랜치로 이동
git switch main
git checkout main                    # 둘 다 됨

# 브랜치 삭제 (머지된 후)
git branch -d feature/login          # 안전 — 머지 안 됐으면 거부
git branch -D feature/login          # 강제 — 작업 통째로 버림
```

### 5.4 브랜치 네이밍 컨벤션

| 접두사 | 용도 | 예 |
|--------|------|-----|
| `feature/` | 새 기능 | `feature/user-login` |
| `fix/`     | 버그 수정 | `fix/null-pointer-on-empty-list` |
| `docs/`    | 문서 | `docs/update-readme` |
| `refactor/`| 리팩토링 | `refactor/auth-module` |
| `chore/`   | 빌드/설정 | `chore/upgrade-deps` |

영어 소문자 + `-` 연결이 관례. 한글 / 공백 / 대문자는 피하자.


---
## 6. Pull Request 흐름 — GitHub Flow

**GitHub Flow** 는 가장 보편적이고 단순한 팀 워크플로. 5단계로 외운다.

```
1. main 최신화        2. 브랜치 + 작업      3. 푸시
        ▼                    ▼                    ▼
   git pull           git switch -c        git push -u origin
                       feature/foo          feature/foo
                       (커밋 여러 번)

   ┌────────────────────────────────────┐
   │      GitHub 웹사이트                │
   │                                    │
   │  4. Pull Request 열기              │
   │  5. 리뷰 / 토론 / 추가 커밋        │
   │  6. Approve → Merge                │
   │                                    │
   └────────────────────────────────────┘
                ▼
        7. 정리 — 로컬도 동기화
        git switch main && git pull
        git branch -d feature/foo
```

### 6.1 PR 열기 (웹에서)

1. push 후 GitHub 저장소 페이지로 가면 **"Compare & pull request"** 노란 띠가 뜸
2. 클릭 → 제목 + 설명 작성 → **Create pull request**

### 6.2 좋은 PR 제목/본문

```
제목: Add user login form with email validation

본문:
## 변경 사항
- 로그인 폼 컴포넌트 추가 (src/auth/LoginForm.tsx)
- 이메일 형식 검증
- 빈 입력값 에러 메시지

## 테스트 방법
1. npm run dev
2. /login 접속
3. 잘못된 이메일 입력 → 에러 확인

## 스크린샷
(붙여넣기)

## 관련 이슈
Closes #42
```

### 6.3 코드 리뷰 받기

리뷰어가 PR 페이지에서:
- **Comment** : 단순 의견 — merge 가능
- **Request changes** : 수정 요청 — 머지 막힘
- **Approve** : 승인 — 머지 가능

수정 요청을 받았다면, **같은 브랜치에 그냥 추가 커밋 후 push** 하면 PR 에 자동 반영된다.

```bash
# 같은 브랜치에 머물러 있는 상태에서
# 코드 수정
git add .
git commit -m "Fix email regex per review"
git push    # 추가 push (-u 없이도 됨, 이미 트래킹 됨)
```

### 6.4 Merge 방식 3가지

GitHub에서 PR Merge 버튼 옆 ▼ 누르면 선택 가능.

| 방식 | 결과 | 언제 |
|------|------|------|
| **Create a merge commit** | 머지 커밋 1개 추가, 모든 히스토리 보존 | 기본값 |
| **Squash and merge** | 브랜치 커밋들을 1개로 압축 | 작은 PR, 깔끔한 히스토리 선호 |
| **Rebase and merge** | 머지 커밋 없이 일렬로 |  히스토리를 직선처럼 유지 |

> 💡 **팀이 한 방식으로 통일**하는 게 중요. 보통 **Squash** 가 깔끔해서 인기.

### 6.5 머지 후 정리

```bash
git switch main
git pull                    # 머지된 변경사항 받기
git branch -d feature/foo   # 로컬 브랜치 삭제

# 리모트 브랜치 자동 삭제 안 됐으면
git push origin --delete feature/foo
```

GitHub 저장소 Settings → General → "Automatically delete head branches" 를 켜두면 머지 후 리모트 브랜치가 자동 삭제됨 — **추천**.


---
## 7. 충돌(merge conflict) 해결

### 7.1 충돌은 왜 생기나

A 와 B 가 **같은 파일의 같은 줄을 다르게 수정** 하고 둘 다 push 하면 git은 자동으로 어느 게 맞는지 판단 못 함.

```
main에서 시작:
    print("Hello")

A의 브랜치:               B의 브랜치:
    print("Hello, World") print("Hello!")

A 가 먼저 머지됨 → main:  print("Hello, World")
B 가 머지 시도 → ❌ 충돌
```

### 7.2 충돌 만났을 때

`git pull` 또는 PR 머지 시 빨간 메시지:
```
CONFLICT (content): Merge conflict in main.py
Automatic merge failed; fix conflicts and then commit the result.
```

해당 파일을 열어보면:
```python
<<<<<<< HEAD
print("Hello, World")
=======
print("Hello!")
>>>>>>> feature/greeting
```

- `<<<<<<< HEAD` ~ `=======` : **현재 브랜치(main)** 의 버전
- `=======` ~ `>>>>>>> feature/greeting` : **들어오려는 브랜치** 의 버전

### 7.3 해결 절차

1. 파일을 열고, 어떻게 합칠지 직접 결정 (둘 다 살리기 / 한쪽 선택 / 새로 작성)
2. **`<<<<<<<`, `=======`, `>>>>>>>` 표시줄을 전부 지움**
3. 저장 → `git add 파일` → `git commit`

```bash
# 충돌 파일 확인
git status                    # both modified: main.py

# 파일 수정 후
git add main.py
git commit                    # 메시지 자동 생성됨, 그대로 저장
```

### 7.4 IDE 도움 받기

VS Code, PyCharm 등은 충돌을 색깔로 표시해주고 **"Accept Current Change"**, **"Accept Incoming Change"**, **"Accept Both"** 버튼을 띄워준다. 처음에는 IDE 도움 받는 게 안전하다.

### 7.5 머지 포기하기

작업이 꼬였다면 머지 작업 전 상태로 돌아갈 수 있다.

```bash
git merge --abort      # 머지 중이었다면
git rebase --abort     # 리베이스 중이었다면
```


---
## 8. 일상 시나리오 8개

실전에서 자주 만나는 상황과 그때의 명령어.

### 시나리오 1. 새 기능 작업 시작

```bash
git switch main
git pull
git switch -c feature/comments
# ... 작업 ...
git add .
git commit -m "Add comment list UI"
git push -u origin feature/comments
```

### 시나리오 2. 작업 중 동료가 main에 큰 변경을 머지함

내 브랜치도 그 변경을 받아 합쳐야 함. 두 방법:

```bash
# (방법 A) 머지 — 안전 / 머지 커밋 생김
git switch feature/comments
git fetch
git merge origin/main

# (방법 B) 리베이스 — 히스토리 깔끔 / 위험도 약간 ↑
git switch feature/comments
git fetch
git rebase origin/main
```

> ⚠️ **이미 push한 브랜치를 rebase 하면** 다른 사람이 받아간 히스토리와 어긋난다. 혼자 작업 중인 브랜치에만 rebase 권장.

### 시나리오 3. 실수로 main 에 직접 커밋했을 때

push 전이면 안전하게 옮길 수 있다.

```bash
git switch -c feature/oops          # 지금 main에서 새 브랜치 만들면 커밋이 따라옴
git switch main
git reset --hard origin/main        # main 은 리모트 상태로 되돌림
git switch feature/oops             # 작업 이어가기
```

### 시나리오 4. 작업 중 다른 급한 일이 끼어들었다 (`stash`)

```bash
# 지금 수정 중인 내용을 임시 저장 (커밋 X)
git stash

# 다른 작업 처리...
git switch hotfix/urgent
# ... 작업 ...

# 원래 작업 돌아오기
git switch feature/comments
git stash pop                       # 임시 저장 꺼내기
```

### 시나리오 5. 잘못된 파일을 add 했다

```bash
# 아직 commit 전이면
git restore --staged file.txt       # add 취소 (파일은 그대로)
git restore file.txt                # 파일 변경 취소 (위험 — 작업 사라짐)
```

### 시나리오 6. 마지막 커밋 메시지를 고치고 싶다

```bash
# 아직 push 안 한 경우만 안전
git commit --amend -m "Fix typo in login form"
```

### 시나리오 7. 커밋을 통째로 취소

```bash
# (push 전) 마지막 커밋만 취소, 변경사항은 staging 에 남김
git reset --soft HEAD~1

# (push 후) 새 "되돌리는" 커밋을 추가 — 안전
git revert <commit-hash>
```

### 시나리오 8. PR 에 리뷰어가 수정 요청을 남겼다

```bash
# 같은 브랜치에 머무른 채로
# 코드 수정...
git add .
git commit -m "Fix per review: handle empty string"
git push                            # PR 페이지에 자동 반영
```


---
## 9. 좋은 커밋 습관

### 9.1 커밋 메시지

좋은 메시지는 **명령형 + 왜를 한 줄로**.

```
✅ Add input validation to login form
✅ Fix race condition in payment retry
✅ Refactor auth module to reduce coupling

❌ updated stuff
❌ aaa
❌ fix
❌ 작업중
```

### 9.2 Conventional Commits (팀 규약 추천)

```
<타입>: <짧은 설명>

[본문 — 왜 변경했는지]

[footer — 관련 이슈 등]
```

| 타입 | 용도 |
|------|------|
| `feat`     | 새 기능 |
| `fix`      | 버그 수정 |
| `docs`     | 문서 |
| `refactor` | 동작 그대로 두고 코드 개선 |
| `test`     | 테스트 추가 |
| `chore`    | 빌드/설정/잡일 |
| `style`    | 포맷팅 (코드 의미 X) |

```
feat: add password strength indicator

비밀번호 입력 시 약/중/강 표시.
사용자 가입률 향상을 위함 (PM 요청).

Closes #123
```

### 9.3 작게 자주 커밋

| 안 좋은 | 좋은 |
|--------|------|
| 한 커밋에 5개 기능 + 버그 수정 + 리팩토링 | 기능 1개 = 커밋 1~3개 |
| "1주일 작업 다 합쳐서 커밋" | 한 단계 끝날 때마다 |
| 작동 안 되는 코드를 main에 | 매 커밋이 빌드는 통과 |

### 9.4 `.gitignore` 에 넣을 것들

언어/환경마다 다르지만 **공통**:

```gitignore
# Python
__pycache__/
*.pyc
.venv/
venv/

# Jupyter
.ipynb_checkpoints/

# Node
node_modules/
dist/

# OS
.DS_Store
Thumbs.db

# IDE
.vscode/
.idea/

# 비밀정보 (제일 중요!)
.env
*.key
credentials.json

# 데이터/모델 (보통 별도 저장소)
data/
*.csv
*.pt
```

GitHub 가 언어별 템플릿 제공: https://github.com/github/gitignore


---
## 10. 자주 하는 실수 & 복구

### 10.1 잘못된 user 로 커밋했을 때

```bash
# 최근 커밋만
git commit --amend --author="New Name <new@email.com>" --no-edit

# 여러 커밋 (위험 — push 전에만)
git rebase -i HEAD~3
```

### 10.2 비밀번호/API 키를 실수로 푸시했다 🚨

**키 자체를 즉시 폐기/재발급** 한 뒤, 히스토리에서 제거:

```bash
# 1) git-filter-repo 설치 후 (간단)
git filter-repo --invert-paths --path config/secrets.json

# 2) 원격 강제 푸시 — 팀에게 알리고 진행
git push --force --all
```

GitHub은 자동 시크릿 스캐닝으로 노출된 토큰을 발견하면 알려주기도 함. 하지만 **이미 노출된 키는 살아있다고 가정** 하고 무조건 재발급.

### 10.3 `git reset --hard` 로 작업이 사라졌다 😱

`reflog` 로 거의 복구 가능. git은 30일 동안 모든 HEAD 이동을 기록한다.

```bash
git reflog                  # 최근 HEAD 이동 목록
# d8a3f12 HEAD@{0}: reset: moving to HEAD~1
# 4b1c8e9 HEAD@{1}: commit: Add feature X    ← 이게 잃어버린 커밋
# ...

git reset --hard 4b1c8e9    # 복구
```

### 10.4 `reset` vs `revert` 언제 뭐?

| 명령 | 동작 | 안전성 |
|------|------|--------|
| `reset --soft HEAD~1`  | 마지막 커밋 취소, 변경 staging에 유지 | push 전만 안전 |
| `reset --hard HEAD~1`  | 마지막 커밋 + 변경사항 모두 삭제 | ⚠️ push 전, 작업 사라짐 |
| `revert <hash>`        | 그 커밋의 반대 동작을 새 커밋으로 추가 | push 후도 안전 |

### 10.5 강제 푸시(`--force`)는 마지막 수단

`git push --force` 는 리모트의 히스토리를 덮어쓴다. 동료가 이미 받아간 커밋이 사라지면 그 사람 작업이 꼬인다.

- ✅ 혼자 작업하는 브랜치
- ✅ 명시적으로 팀 합의된 경우
- ❌ `main` 같은 공용 브랜치 — **절대 금지**

`--force-with-lease` 가 그나마 안전 (다른 사람이 그 사이 push 했으면 거부됨).

```bash
git push --force-with-lease
```


---
## 11. 협업 에티켓

기술 못지않게 **사람** 이 중요한 부분.

### 11.1 PR 작은 단위로 쪼개기

| 안 좋은 | 좋은 |
|--------|------|
| PR 한 개에 +2000줄, 파일 30개 | PR 한 개에 ~300줄 이내 |
| "리뷰어가 알아서 보겠지" | 리뷰 받기 쉽게 컨텍스트 설명 |
| 무관한 변경 섞임 (기능 + 포맷팅 + 리팩토링) | 한 PR = 한 주제 |

> 💡 큰 작업은 **여러 PR 로 쪼개기**. 예: 1) DB 스키마, 2) API, 3) UI

### 11.2 좋은 PR 설명 템플릿

```markdown
## What
이 PR이 무엇을 바꾸는지 (2~3줄)

## Why
왜 필요한지 (배경, 관련 이슈)

## How
어떻게 구현했는지 (선택사항 — 비자명할 때만)

## Test plan
- [ ] 단계 1
- [ ] 단계 2

## Screenshots
(UI 변경 시)
```

### 11.3 리뷰어의 태도

- ✅ "이 부분 `setTimeout` 보다 `requestAnimationFrame` 이 더 적합할 것 같아요. 이유는 ..."
- ✅ "여기 의도가 좀 헷갈리는데 주석 추가해줄 수 있을까요?"
- ❌ "이게 뭐야 이렇게 짜면 안 되지"
- ❌ (커밋 후 6개월 뒤) "이거 누가 짠 거야?"

**코드를 비판하되 사람을 비판하지 않는다.**

### 11.4 리뷰어가 안 보이면

이틀째 리뷰가 없으면 가볍게 핑(@멘션). 리뷰는 **48시간 이내** 가 일반적 SLA.

### 11.5 main 브랜치 보호하기

GitHub 저장소 Settings → **Branches** → Add rule for `main`:
- ✅ Require a pull request before merging
- ✅ Require approvals (최소 1명)
- ✅ Dismiss stale approvals when new commits are pushed
- ✅ Require status checks (CI 통과)
- ⛔ Disable force push

이걸 켜두면 `main` 에 실수로 push 자체가 막혀서 사고 예방.


---
## ✏️ 연습문제

**Q1.** 다음 상황에서 무엇이 잘못됐고 어떻게 고쳐야 할까?

> 신입 개발자 A가 `main` 브랜치에서 바로 새 기능을 개발했다. 한 달 작업을 한 번에 `git commit -m "작업 완료"` 로 커밋하고 push 했다. 그 사이 동료들이 `main` 에 머지한 변경이 많아 충돌이 폭발했다.

**Q2.** 다음 명령어들이 각각 무엇을 하는지 한 줄로 설명:
```bash
git fetch
git pull
git merge
git rebase
git revert
git reset --soft HEAD~1
git stash
git switch -c feature/foo
```

**Q3.** 다음을 실제로 해본다 (실습):
1. `IntroductiontoAI` 저장소를 clone
2. `feature/my-name` 브랜치 생성
3. README 끝에 자기 이름 한 줄 추가
4. commit & push
5. GitHub에서 PR 열기
6. 자기가 자기 PR 머지
7. 로컬 main 동기화 + 브랜치 삭제

**Q4.** 동료의 PR을 리뷰한다고 가정. 다음 PR 설명의 어떤 점이 부족한지 비판해보자:

```
제목: 수정
본문: 버그 고침
```

---

## 🎯 핵심 요약 — 외울 4개 명령

```bash
git pull                              # 시작 전 최신화
git switch -c feature/xxx             # 브랜치 만들기
git add . && git commit -m "..."      # 커밋
git push -u origin feature/xxx        # 푸시 → GitHub 에서 PR
```

나머지는 필요할 때 검색해서 익히면 된다. 가장 중요한 건 **`main` 에서 직접 작업하지 않기** 와 **자주 pull 하기**.
